# 14 — Land-use presentation analyses for the Results chapter

**Purpose.** Present the model's restoration indicators and the Restoration Indicator Score (RIS) by land cover, at pixel level and at burned-zone level, and compare the reference fire inventory with the model's own detection — on the existing 30 m full-region inference rasters. No model is retrained; the only external input that may be needed is a 2018 land-cover layer on the inference grid (see §2).

**Framing that applies to every output here (from Notebook 13).** Cross-class differences in the recovery gap and the other NBR-derived indicators largely reflect the baseline normalisation of the index, not ecological recovery: the recovery-gap target divides by each pixel's 2017 NBR level, which differs about two-fold between cropland and forest while their absolute NBR loss is similar, and NBR measured on each pixel's own fire shows no class ordering once calendar period is held fixed. The tables and maps below are therefore for honest presentation and for comparing detection, **not** evidence of land-cover-dependent ecological recovery. Captions repeat this.

**Definitions** (verified in §2 where they can be):

| item | definition | source |
|---|---|---|
| indicators | model outputs at 30 m: severity, persistence, recovery gap (regression heads); recurrence = model-derived `min(count of months with burned_prob > threshold, 5)/5` | `outputs/inference_full_roi/{severity,persistence,recovery_gap,model_recurrence}_30m.tif` |
| RIS | `w_s·severity + w_p·persistence + w_r·recovery_gap + w_c·recurrence`; profiles Ecological (0.20, 0.20, 0.50, 0.10), Fire-focused (0.35, 0.35, 0.20, 0.10), Balanced (0.25 each). Recurrence is the **model-derived** layer — the same definition as the thesis RIS maps (checked in §2) | the four indicator rasters |
| eligible pixel | finite in all indicator rasters (the eco mask used at inference) | indicator rasters |
| documented-burned | eligible pixel whose centre lies inside a fire polygon dated 2017-2025 (the label window; the 92 polygons dated 2016 are excluded) | `data/thesis_maps/fig_gt_fires.shp` |
| model-detected | eligible pixel with `burned_prob > 0.502` (burned_prob stores the maximum over the 108 monthly probabilities) | `burned_prob_30m.tif` |
| burned zone | 8-connected component of the documented-burned mask | derived here |
| land cover | 2018 `landcover_smoothed_yearly`, band `b1`, mode-aggregated to the 30 m inference grid | `data/raw/landcover_2018_30m.tif` (export: `notebooks/export_landcover_2018_30m.py`) |

**Contents.** §2 input audit and checks · §3 RIS layers, masks, burned zones · §4 deliverable 1: RIS per land-cover class · §5 deliverable 2: regional hierarchy (zone means → class) · §6 deliverable 3: reference vs model-detected priority maps · §7 deliverable 4: per-class small-multiple maps · §8 registry and output notes.

Every table's Markdown copy lists the raster or array behind each column; every output prints a one-line `NOTE` with what it shows and its key numbers. Files are prefixed `nb14_`.

In [ ]:
import hashlib
import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.transform import Affine
from rasterio.warp import transform as warp_transform
from scipy import ndimage
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon
from shapely.ops import unary_union
from shapely.validation import make_valid
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = os.environ.get("THESIS_ROOT")
if not ROOT:
    here = os.path.abspath(os.getcwd())
    ROOT = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
ROOT = os.path.abspath(os.path.expanduser(ROOT))

RAST = os.path.join(ROOT, "outputs", "inference_full_roi")
FIRES_SHP = os.path.join(ROOT, "data", "thesis_maps", "fig_gt_fires.shp")
ROI_SHP = os.path.join(ROOT, "data", "thesis_maps", "fig_roi_polygon.shp")
THESIS_RIS = os.path.join(ROOT, "data", "thesis_maps", "fig_ris_balanced.tif")
NB13_TAB = os.path.join(ROOT, "diagnostics", "nb13_landcover_timing", "tables")
LC_PATH = os.environ.get("NB14_LC_RASTER", os.path.join(ROOT, "data", "raw", "landcover_2018_30m.tif"))

# Outputs go to the thesis output folders. NB14_OUT_ROOT redirects them for code tests only.
OUT_ROOT = os.path.abspath(os.environ.get("NB14_OUT_ROOT", ROOT))
TEST_MODE = OUT_ROOT != ROOT
if "NB14_LC_RASTER" in os.environ:
    assert TEST_MODE and os.path.commonpath([OUT_ROOT, ROOT]) != ROOT, \
        "a substitute land-cover raster may only be used with NB14_OUT_ROOT outside the project"
TAB = os.path.join(OUT_ROOT, "outputs", "tables")
FIG = os.path.join(OUT_ROOT, "outputs", "figures")
os.makedirs(TAB, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print("thesis root :", ROOT)
print("writes to   :", OUT_ROOT + "/outputs/{tables,figures}")
if TEST_MODE:
    print("*** TEST MODE — outputs redirected; land-cover raster:", LC_PATH, "***")
print("run (UTC)   :", RUN_STAMP)
print("python      :", sys.version.split()[0], "|", platform.platform())
print("numpy", np.__version__, "| pandas", pd.__version__, "| rasterio", rasterio.__version__,
      "| geopandas", gpd.__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
# ---- constants and helpers ------------------------------------------------
LC_NAMES = {10: "forest", 11: "orchards", 20: "shrubland", 30: "grassland",
            40: "cropland", 41: "covered agriculture", 50: "built-up",
            60: "bare", 61: "fallow", 80: "water"}
LC_ORDER = [10, 11, 20, 30, 40, 41, 50, 60, 61, 80]
OTHER = "unclassified / other code"
CLASS_NAMES = [LC_NAMES[c] for c in LC_ORDER] + [OTHER]
_cmap = plt.get_cmap("tab10")
CLASS_COLOR = {LC_NAMES[c]: _cmap(i) for i, c in enumerate(LC_ORDER)}
CLASS_COLOR[OTHER] = (0.6, 0.6, 0.6, 1.0)

PROFILES = {"ecological": (0.20, 0.20, 0.50, 0.10),
            "fire": (0.35, 0.35, 0.20, 0.10),
            "balanced": (0.25, 0.25, 0.25, 0.25)}
PROFILE_LABEL = {"ecological": "Ecological priority", "fire": "Fire-focused", "balanced": "Balanced"}
IND = ["severity", "persistence", "recovery_gap", "model_recurrence"]

GRID_T = Affine(30.0, 0.0, 620610.0, 0.0, -30.0, 3514560.0)
H, W, CRS = 2598, 1625, "EPSG:32636"
PX_HA = 0.09                # one 30 m pixel
THR = 0.502                 # detection threshold on burned_prob
MIN_PX_CLASS = 100          # 9 ha: classes below this are flagged small-n and left out of figures
MIN_UNIT_PX = 4             # 0.36 ha: smallest zone (or zone x class unit) averaged in §5 (~0.3 ha as in Notebook 11)
RESTORATION_CMAP = LinearSegmentedColormap.from_list(   # the thesis Figure 4.2/4.3 colormap
    "restoration", ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"], N=256)
FRAMING = ("Cross-class differences largely reflect the NBR baseline normalization, "
           "not ecological recovery.")

PREFIX = "nb14_"
REGISTRY, NOTES = [], []

matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                            "axes.spines.top": False, "axes.spines.right": False})


def _fingerprint(path, edge=16 * 1024 * 1024):
    h = hashlib.sha256()
    size = os.path.getsize(path)
    with open(path, "rb") as fh:
        h.update(fh.read(min(edge, size)))
        if size > edge:
            fh.seek(max(0, size - edge))
            h.update(fh.read(edge))
    return h.hexdigest()[:16], size


def _markdown_table(df, fmt="{:.4f}"):
    def cell(v):
        if isinstance(v, (bool, np.bool_)):
            return str(bool(v))
        if isinstance(v, (float, np.floating)):
            return "" if not np.isfinite(v) else fmt.format(v)
        return str(v)
    cols = list(df.columns)
    out = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(out) + "\n"


def _rel(path):
    return os.path.relpath(path, OUT_ROOT)


def _header(section):
    return "<!-- generated by notebooks/14_landuse_presentation.ipynb §{} on {} -->\n\n".format(section, RUN_STAMP)


def save_table(df, stem, caption, section, sources):
    name = PREFIX + stem
    df.to_csv(os.path.join(TAB, name + ".csv"), index=False)
    with open(os.path.join(TAB, name + ".md"), "w") as fh:
        fh.write(_header(section) + caption.rstrip() + "\n\n**Sources (column → raster / array).**\n\n")
        fh.write("\n".join("- `{}` — {}".format(k, v) for k, v in sources.items()) + "\n\n")
        fh.write(_markdown_table(df))
    REGISTRY.append(dict(section=str(section), kind="table", stem=name,
                         files=_rel(os.path.join(TAB, name)) + ".csv, .md",
                         sources="; ".join(sorted(set(sources.values()))), caption=caption))
    print("  ->", _rel(os.path.join(TAB, name + ".csv")), "(+ .md)")


def save_figure(fig, stem, caption, section, sources):
    name = PREFIX + stem
    path = os.path.join(FIG, name + ".png")
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    with open(os.path.join(FIG, name + ".md"), "w") as fh:
        fh.write(_header(section) + caption.rstrip() + "\n\n**Sources.**\n\n")
        fh.write("\n".join("- `{}` — {}".format(k, v) for k, v in sources.items()) + "\n")
    REGISTRY.append(dict(section=str(section), kind="figure", stem=name,
                         files=_rel(path) + ", .md", sources="; ".join(sorted(set(sources.values()))),
                         caption=caption))
    print("  ->", _rel(path))


def output_note(stem, text):
    line = "NOTE [{}{}] {}".format(PREFIX, stem, text)
    NOTES.append(line)
    print(line)


def km_axes(ax):
    fmt = FuncFormatter(lambda v, _: "{:.0f}".format(v / 1000))
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)
    ax.set_xlabel("UTM 36N easting (km)")
    ax.set_ylabel("northing (km)")

## 2. Input audit and checks

**Question.** Which of the required inputs exist, are they on the inference grid, and do the definitions above hold?

**Why.** Every number below is read from these files. The land-cover raster in particular is an external export: it is only usable if it sits exactly on the 1625 × 2598, 30 m inference grid, and the notebook refuses it otherwise.

**Data.** The six rasters in `outputs/inference_full_roi/`, the fire-reference polygons, the land-cover raster (if present), the thesis's own RIS map `data/thesis_maps/fig_ris_balanced.tif` (used only to confirm the RIS definition) and the Notebook 13 tables reused in §4.

**Metric.** Presence, grid (shape, CRS, transform), finite-pixel count, range and a head+tail SHA-256 per raster. Checks: all indicator rasters share one grid and one eligible footprint; `burned_prob > 0.502` selects the same pixels as model recurrence ≥ 0.2; the thesis Balanced-RIS map equals `0.25·(sev + per + rg + model_recurrence)` at 5,000 random detected pixels (and does not equal the version with reference recurrence).

**Reading the output.** Grid and footprint checks are asserted. If the land-cover raster is missing or misaligned, §4, §5 and §7 print SKIPPED and name the export script.

In [ ]:
MODEL_LAYERS = {"severity": "severity_30m.tif", "persistence": "persistence_30m.tif",
                "recovery_gap": "recovery_gap_30m.tif", "model_recurrence": "model_recurrence_30m.tif",
                "burned_prob": "burned_prob_30m.tif"}
OTHER_LAYERS = {"recurrence_reference": "recurrence_30m.tif"}
A, rows = {}, []
for role, fn in {**MODEL_LAYERS, **OTHER_LAYERS}.items():
    path = os.path.join(RAST, fn)
    if not os.path.exists(path):
        rows.append(dict(input=os.path.relpath(path, ROOT), role=role, present=False, detail="MISSING"))
        continue
    with rasterio.open(path) as s:
        a = s.read(1).astype(np.float64)
        shape_, crs_, tr_ = (s.height, s.width), str(s.crs), s.transform
    fp, size = _fingerprint(path)
    A[role] = a
    rows.append(dict(input=os.path.relpath(path, ROOT), role=role, present=True,
                     rows_x_cols="{} x {}".format(*shape_), crs=crs_, pixel_m=tr_.a,
                     on_inference_grid=bool(shape_ == (H, W) and crs_ == CRS and tr_.almost_equals(GRID_T)),
                     finite_px=int(np.isfinite(a).sum()), min=float(np.nanmin(a)),
                     mean=float(np.nanmean(a)), max=float(np.nanmax(a)), sha256_head_tail=fp,
                     detail="reference recurrence from the 10 m burn-count raster (context only)"
                     if role == "recurrence_reference" else "model output (Notebook 07b)"))
assert all(k in A for k in MODEL_LAYERS), "an indicator raster is missing"

fires = gpd.read_file(FIRES_SHP)
fp, _ = _fingerprint(FIRES_SHP)
rows.append(dict(input=os.path.relpath(FIRES_SHP, ROOT), role="reference fire polygons", present=True,
                 crs=str(fires.crs), sha256_head_tail=fp,
                 detail="{:,} polygons: {:,} dated 2017-2025 (used), {:,} dated 2016 (outside the label window, "
                        "excluded); sources {}".format(len(fires), int((fires.fire_year >= 2017).sum()),
                                                        int((fires.fire_year < 2017).sum()),
                                                        fires.source.value_counts().to_dict())))

HAVE_LC, LCR, lc_detail = False, None, ""
if os.path.exists(LC_PATH):
    with rasterio.open(LC_PATH) as s:
        LCR = s.read(1)
        lc_shape, lc_crs, lc_tr, lc_nodata = (s.height, s.width), str(s.crs), s.transform, s.nodata
    lc_on_grid = bool(lc_shape == (H, W) and lc_crs == CRS and lc_tr.almost_equals(GRID_T))
    HAVE_LC = lc_on_grid and np.issubdtype(LCR.dtype, np.integer)
    fp, _ = _fingerprint(LC_PATH)
    lc_detail = "dtype {}, nodata {}; {}".format(LCR.dtype, lc_nodata,
                                                 "aligned" if lc_on_grid else "NOT on the inference grid — refused")
    rows.append(dict(input=LC_PATH if TEST_MODE else os.path.relpath(LC_PATH, ROOT),
                     role="land cover 2018, 30 m mode" + (" (TEST PROXY)" if TEST_MODE else ""), present=True,
                     rows_x_cols="{} x {}".format(*lc_shape), crs=lc_crs, pixel_m=lc_tr.a,
                     on_inference_grid=lc_on_grid, sha256_head_tail=fp, detail=lc_detail))
else:
    rows.append(dict(input=os.path.relpath(LC_PATH, ROOT), role="land cover 2018, 30 m mode", present=False,
                     detail="MISSING — export with notebooks/export_landcover_2018_30m.py"))

for label, path, detail in [("thesis Balanced-RIS map (definition check only)", THESIS_RIS, "EPSG:4326 reprojection used for Figure 4.3"),
                            ("Notebook 13 per-class tables (reused in §4)", NB13_TAB, "")]:
    ok = os.path.exists(path)
    rows.append(dict(input=os.path.relpath(path, ROOT), role=label, present=ok,
                     detail=detail if ok else "MISSING"))
inventory = pd.DataFrame(rows)
display(inventory)

# ---- checks -----------------------------------------------------------------
ELIG = np.isfinite(A["severity"])
footprint_same = all(np.array_equal(np.isfinite(A[k]), ELIG) for k in MODEL_LAYERS)
det_a = A["burned_prob"][ELIG] > THR
det_b = A["model_recurrence"][ELIG] >= 0.2 - 1e-6

rng = np.random.default_rng(42)
r_, c_ = np.nonzero(ELIG & (A["burned_prob"] > THR))
k = rng.choice(len(r_), size=min(5000, len(r_)), replace=False)
xs, ys = rasterio.transform.xy(GRID_T, r_[k], c_[k])
lon, lat = warp_transform(CRS, "EPSG:4326", list(xs), list(ys))
with rasterio.open(THESIS_RIS) as s:
    v = np.array([x[0] for x in s.sample(zip(lon, lat))], dtype=np.float64)
base = (A["severity"] + A["persistence"] + A["recovery_gap"])[r_[k], c_[k]]
fin = np.isfinite(v)
share_model_rec = float(np.mean(np.abs(v[fin] - 0.25 * (base + A["model_recurrence"][r_[k], c_[k]])[fin]) < 1e-3))
share_ref_rec = float(np.mean(np.abs(v[fin] - 0.25 * (base + np.nan_to_num(A["recurrence_reference"][r_[k], c_[k]]))[fin]) < 1e-3))

checks = pd.DataFrame([
    dict(check="all five indicator rasters on the inference grid (2598 rows x 1625 cols, EPSG:32636, 30 m)",
         value=str(bool(all(r.get("on_inference_grid") for r in rows if r["role"] in MODEL_LAYERS))),
         passed=bool(all(r.get("on_inference_grid") for r in rows if r["role"] in MODEL_LAYERS))),
    dict(check="identical eligible (finite) footprint across the five rasters",
         value="{:,} eligible pixels".format(int(ELIG.sum())), passed=bool(footprint_same)),
    dict(check="burned_prob > 0.502 selects the same pixels as model_recurrence >= 0.2",
         value="agreement {:.6f}".format(float(np.mean(det_a == det_b))), passed=bool(np.array_equal(det_a, det_b))),
    dict(check="thesis Balanced-RIS map = 0.25(sev+per+rg+MODEL recurrence) at sampled detected pixels",
         value="{:.4f} within 1e-3 (reference-recurrence version: {:.4f})".format(share_model_rec, share_ref_rec),
         passed=bool(share_model_rec > 0.99 and share_model_rec > share_ref_rec)),
    dict(check="2018 land-cover raster present and aligned to the inference grid",
         value=lc_detail if lc_detail else "missing", passed=bool(HAVE_LC)),
])
display(checks)
assert checks.loc[:1, "passed"].all(), "indicator rasters are not usable — stop"
save_table(inventory, "t00_input_audit",
           "Input audit for Notebook 14: presence, grid, value range and head+tail SHA-256 of every input.",
           section=2, sources={"all columns": "file metadata and pixel values of each listed input"})
save_table(checks, "t00b_input_checks",
           "Checks on the inputs and on the RIS definition used throughout Notebook 14.", section=2,
           sources={"grid / footprint": "outputs/inference_full_roi/*_30m.tif",
                    "detection equivalence": "burned_prob_30m.tif, model_recurrence_30m.tif",
                    "RIS definition": "data/thesis_maps/fig_ris_balanced.tif vs severity/persistence/recovery_gap/model_recurrence/recurrence _30m.tif",
                    "land cover": "data/raw/landcover_2018_30m.tif"})
output_note("t00_input_audit", "{} of {} inputs present; indicator rasters on grid with {:,} eligible 30 m pixels; "
            "land-cover raster {}; thesis RIS uses model recurrence ({:.1%} match vs {:.1%} with reference recurrence)."
            .format(int(inventory.present.sum()), len(inventory), int(ELIG.sum()),
                    "PRESENT and aligned" if HAVE_LC else "MISSING", share_model_rec, share_ref_rec))
if not HAVE_LC:
    print("\n" + "!" * 88)
    print("LAND-COVER RASTER MISSING OR MISALIGNED:", LC_PATH)
    print("Deliverables 1, 2 and 4 (§4, §5, §7) are skipped. Run notebooks/export_landcover_2018_30m.py,")
    print("download landcover_2018_30m.tif to data/raw/, and re-run this notebook.")
    print("!" * 88)

## 3. RIS layers, detection masks and burned zones

**Question.** Where are the documented-burned and model-detected pixels, how far do they agree, and what are the burned zones that §5 averages over?

**Why.** Deliverables 1-4 all use these three objects. The two masks differ substantially, so every land-cover result is reported under both.

**Data.** The four indicator rasters and `burned_prob_30m.tif`; `fig_gt_fires.shp` (polygons dated 2017-2025, rasterised at 30 m pixel centres); `recurrence_30m.tif` as an alternative reference mask for context only.

**Metric.** Pixel counts and areas of each mask, their overlap (Jaccard index), the share of reference pixels the model detects, the share of detected pixels inside the polygons, and Balanced-RIS mean/median in each part. Zones: 8-connected components of the documented-burned mask; their size distribution and per-zone indicator means.

**Reading the output.** The zone inventory is land-cover independent and is written even without the land-cover raster.

In [ ]:
SEV, PER, RG, MREC, BP = (A[k] for k in ("severity", "persistence", "recovery_gap", "model_recurrence", "burned_prob"))
RIS = {p: np.where(ELIG, ws * SEV + wp * PER + wr * RG + wc * MREC, np.nan) for p, (ws, wp, wr, wc) in PROFILES.items()}
Q = {"severity": SEV, "persistence": PER, "recovery_gap": RG, "model_recurrence": MREC}
Q.update({"RIS_" + p: RIS[p] for p in PROFILES})
QUANT = list(Q)


def _polygonal(g):
    if g is None or g.is_empty:
        return None
    g = make_valid(g)
    if isinstance(g, (Polygon, MultiPolygon)):
        return g
    if isinstance(g, GeometryCollection):
        parts = [p for p in g.geoms if isinstance(p, (Polygon, MultiPolygon))]
        return unary_union(parts) if parts else None
    return None


fires_win = fires[fires.fire_year >= 2017].to_crs(CRS)
geoms = [g for g in (_polygonal(g) for g in fires_win.geometry) if g is not None and not g.is_empty]
INSIDE = features.rasterize(((g, 1) for g in geoms), out_shape=(H, W), transform=GRID_T,
                            fill=0, all_touched=False, dtype="uint8").astype(bool)
REF = INSIDE & ELIG
DET = ELIG & (BP > THR)
REF_ALT = ELIG & (A["recurrence_reference"] > 0)
BOTH = REF & DET


def _ris_stats(m):
    v = RIS["balanced"][m]
    return (float(v.mean()), float(np.median(v))) if v.size else (np.nan, np.nan)


rows = []
for item, m, detail in [
        ("eligible pixels (indicators defined)", ELIG, "finite in all indicator rasters"),
        ("inside 2017-2025 fire polygons, any pixel", INSIDE, "rasterised at pixel centres"),
        ("inside polygons but not eligible (no indicators)", INSIDE & ~ELIG, "outside the inference eco mask"),
        ("(a) documented-burned", REF, "inside polygons and eligible"),
        ("(b) model-detected", DET, "burned_prob > 0.502"),
        ("both (a) and (b)", BOTH, ""),
        ("documented-burned, not detected", REF & ~DET, "missed by the model"),
        ("detected, outside reference polygons", DET & ~REF, "not in the incomplete inventory"),
        ("context: recurrence_30m > 0 (alternative reference mask)", REF_ALT, "10 m burn count read at 30 m")]:
    n = int(m.sum())
    mu, md = _ris_stats(m) if m is not INSIDE and m is not ELIG else (np.nan, np.nan)
    if item.startswith("inside polygons but not"):
        mu, md = np.nan, np.nan
    rows.append(dict(item=item, n_px=n, area_ha=n * PX_HA, mean_RIS_balanced=mu, median_RIS_balanced=md, value=np.nan, detail=detail))
jac = BOTH.sum() / (REF | DET).sum()
rows += [dict(item="Jaccard index, (a) vs (b)", value=float(jac), detail=""),
         dict(item="% of documented-burned pixels detected", value=100 * float(BOTH.sum() / REF.sum()), detail=""),
         dict(item="% of detected pixels inside reference polygons", value=100 * float(BOTH.sum() / DET.sum()), detail=""),
         dict(item="Jaccard index, (a) vs recurrence_30m > 0", value=float((REF & REF_ALT).sum() / (REF | REF_ALT).sum()),
              detail="context: sensitivity of the reference mask to its construction")]
agreement = pd.DataFrame(rows)
display(agreement)
SRC_M = {"documented-burned": "data/thesis_maps/fig_gt_fires.shp (fire_year 2017-2025) rasterised on the grid of severity_30m.tif",
         "eligible": "finite pixels of severity/persistence/recovery_gap/model_recurrence/burned_prob _30m.tif",
         "model-detected": "outputs/inference_full_roi/burned_prob_30m.tif > 0.502",
         "RIS_balanced": "0.25 x (severity + persistence + recovery_gap + model_recurrence) _30m.tif",
         "context mask": "outputs/inference_full_roi/recurrence_30m.tif"}
save_table(agreement, "t02_reference_vs_detected_masks",
           "Reference-burned versus model-detected pixels at 30 m: counts, areas, overlap and Balanced RIS in "
           "each part. The reference inventory is incomplete, so 'detected, outside reference polygons' is not "
           "the same as false detection.", section=3, sources=SRC_M)
output_note("t02_reference_vs_detected_masks",
            "documented-burned {:,} px ({:,.0f} ha) vs model-detected {:,} px ({:,.0f} ha); overlap {:,} px, Jaccard {:.3f}; "
            "{:.1f}% of reference detected, {:.1f}% of detections inside polygons."
            .format(int(REF.sum()), REF.sum() * PX_HA, int(DET.sum()), DET.sum() * PX_HA, int(BOTH.sum()), jac,
                    100 * BOTH.sum() / REF.sum(), 100 * BOTH.sum() / DET.sum()))

# ---- burned zones -------------------------------------------------------------
ZONE, N_ZONES = ndimage.label(REF, structure=np.ones((3, 3), dtype=int))
ZID = ZONE[REF]                                   # row-major order, same as np.nonzero(REF)
zsize = np.bincount(ZID, minlength=N_ZONES + 1)[1:]
rr, cc = np.nonzero(REF)
zx = GRID_T.c + (cc + 0.5) * GRID_T.a
zy = GRID_T.f + (rr + 0.5) * GRID_T.e
zones = pd.DataFrame(dict(zone_id=np.arange(1, N_ZONES + 1), n_px=zsize, area_ha=zsize * PX_HA,
                          centroid_x=np.bincount(ZID, weights=zx)[1:] / zsize,
                          centroid_y=np.bincount(ZID, weights=zy)[1:] / zsize,
                          pct_detected=100 * np.bincount(ZID, weights=DET[REF].astype(float))[1:] / zsize,
                          meets_min_unit_px=zsize >= MIN_UNIT_PX))
for q in QUANT:
    zones["mean_" + q] = np.bincount(ZID, weights=Q[q][REF])[1:] / zsize
save_table(zones, "t03_burned_zone_inventory",
           "Burned zones = 8-connected components of the documented-burned mask, with per-zone means of every "
           "indicator and RIS profile. Zones below {} pixels ({:.2f} ha) are flagged and not averaged in §5."
           .format(MIN_UNIT_PX, MIN_UNIT_PX * PX_HA), section=3,
           sources={"zone_id / n_px / centroid": "8-connected components of the documented-burned mask (fig_gt_fires.shp on the 30 m grid)",
                    "mean_<indicator>": "outputs/inference_full_roi/<indicator>_30m.tif averaged over zone pixels",
                    "mean_RIS_*": "RIS computed from the four indicator rasters",
                    "pct_detected": "burned_prob_30m.tif > 0.502"})
big = zones[zones.meets_min_unit_px]
output_note("t03_burned_zone_inventory",
            "{:,} burned zones; median {:.0f} px ({:.2f} ha), largest {:,} px; {:,} zones >= {} px hold {:.1f}% of documented-burned pixels."
            .format(N_ZONES, zones.n_px.median(), zones.n_px.median() * PX_HA, int(zones.n_px.max()),
                    len(big), MIN_UNIT_PX, 100 * big.n_px.sum() / zones.n_px.sum()))

# LC helper arrays, used by §4, §5, §7
if HAVE_LC:
    LC_CODE = LCR.astype(np.int32)
    CLS = np.full(LC_CODE.shape, len(LC_ORDER), dtype=np.int16)       # index into CLASS_NAMES; default = other
    for i, c in enumerate(LC_ORDER):
        CLS[LC_CODE == c] = i

## 4. Deliverable 1 — RIS per land-cover class

**Question.** What are the mean and median RIS, under each of the three weight profiles, per 2018 land-cover class — over documented-burned pixels and over model-detected pixels?

**Why.** This is the land-use summary for the Results chapter. It is computed under both masks because they share only about a fifth of their pixels (§3), and a class summary conditioned on the model's own detection can differ from one conditioned on the inventory.

**Data.** RIS from the four indicator rasters (§3); `landcover_2018_30m.tif`; the two masks from §3. Normalisation context is **reused from Notebook 13** rather than recomputed: `nb13_t04a` gives each class's recovery-gap *target* mean, its share clipped at 1, and its mean 2017 NBR (the target's denominator) on the 10 m burned sample.

**Metric.** Per class and population: pixel count, area, share of the population, RIS mean and median per profile, and the mean of each indicator (used again in §5). Classes with fewer than 100 pixels (9 ha) are flagged `small_n` and left out of the figure. Pixels whose land-cover code is not one of the ten classes are grouped as "unclassified / other code"; table t01a lists every code found.

**Reading the output.** Compare classes within a population, and a class across populations. The Notebook 13 columns are on a different unit (10 m sample pixels, modal 2018-2025 class) and are context for why class means differ, not values to compare numerically. Cross-class RIS differences inherit the normalisation artefact of the recovery gap, which carries half the weight in the Ecological profile.

In [ ]:
if not HAVE_LC:
    print("SKIPPED — deliverable 1 needs data/raw/landcover_2018_30m.tif aligned to the inference grid (see §2).")
else:
    code_rows = []
    vals, counts = np.unique(LC_CODE[ELIG], return_counts=True)
    for vcode, n in zip(vals, counts):
        m = LC_CODE == vcode
        code_rows.append(dict(code=int(vcode), land_cover=LC_NAMES.get(int(vcode), OTHER), n_px_eligible=int(n),
                              n_px_reference_burned=int((m & REF).sum()), n_px_model_detected=int((m & DET).sum())))
    lc_codes = pd.DataFrame(code_rows)
    display(lc_codes)

    POPS = [("documented-burned", REF), ("model-detected", DET)]
    rows = []
    for pop, M in POPS:
        npop = int(M.sum())
        for ci, cname in list(enumerate(CLASS_NAMES)) + [(None, "all classes")]:
            m = M if ci is None else (M & (CLS == ci))
            n = int(m.sum())
            if n == 0:
                continue
            row = dict(population=pop, land_cover=cname, n_px=n, area_ha=n * PX_HA,
                       pct_of_population=100.0 * n / npop, small_n=bool(ci is not None and n < MIN_PX_CLASS))
            for p in PROFILES:
                v = RIS[p][m]
                row["RIS_%s_mean" % p] = float(v.mean())
                row["RIS_%s_median" % p] = float(np.median(v))
            for q in IND:
                row[q + "_mean"] = float(Q[q][m].mean())
            rows.append(row)
    ris_tab = pd.DataFrame(rows)
    ref_order = (ris_tab[(ris_tab.population == "documented-burned") & ~ris_tab.land_cover.isin(["all classes", OTHER])]
                 .sort_values("RIS_balanced_mean", ascending=False).land_cover.tolist())
    order = (ref_order + [c for c in ris_tab.land_cover.unique() if c not in ref_order + [OTHER, "all classes"]]
             + [OTHER, "all classes"])
    ris_tab["_o"] = ris_tab.land_cover.map({c: i for i, c in enumerate(order)})
    ris_tab["_p"] = ris_tab.population.map({"documented-burned": 0, "model-detected": 1})
    ris_tab = ris_tab.sort_values(["_p", "_o"]).drop(columns=["_o", "_p"]).reset_index(drop=True)

    nb13_file = os.path.join(NB13_TAB, "nb13_t04a_recovery_gap_distribution_by_class.csv")
    if os.path.exists(nb13_file):
        ctx = (pd.read_csv(nb13_file)[["land_cover", "mean", "pct_at_1", "mean_NBR_2017"]]
               .rename(columns={"mean": "nb13_rg_target_mean_10m", "pct_at_1": "nb13_rg_target_pct_at_1_10m",
                                "mean_NBR_2017": "nb13_mean_NBR_2017_10m"}))
        ris_tab = ris_tab.merge(ctx, on="land_cover", how="left")
    else:
        print("Notebook 13 table not found — context columns omitted:", nb13_file)
    display(ris_tab)

    SRC_1 = {"population": "documented-burned: fig_gt_fires.shp (2017-2025) on the 30 m grid; model-detected: burned_prob_30m.tif > 0.502",
             "land_cover": "data/raw/landcover_2018_30m.tif (landcover_smoothed_yearly 2018, b1, 30 m mode)",
             "RIS_<profile>_mean / _median": "weighted sum of severity_30m.tif, persistence_30m.tif, recovery_gap_30m.tif, model_recurrence_30m.tif",
             "<indicator>_mean": "outputs/inference_full_roi/<indicator>_30m.tif",
             "nb13_*": "diagnostics/nb13_landcover_timing/tables/nb13_t04a_recovery_gap_distribution_by_class.csv (10 m burned sample; reused, not recomputed)"}
    save_table(lc_codes, "t01a_landcover_codes",
               "Land-cover codes found in the 2018 30 m layer over eligible pixels, with their documented-burned and "
               "model-detected pixel counts.", section=4,
               sources={"code / counts": "data/raw/landcover_2018_30m.tif, masks from §3"})
    save_table(ris_tab, "t01_ris_by_landcover",
               "Restoration Indicator Score per 2018 land-cover class under three weight profiles, over "
               "documented-burned and over model-detected 30 m pixels (mean and median), with indicator means and "
               "Notebook 13 normalisation context. " + FRAMING, section=4, sources=SRC_1)

    show = [c for c in order if c != "all classes"
            and len(ris_tab[(ris_tab.land_cover == c) & ~ris_tab.small_n]) > 0]
    POP_COLOR = {"documented-burned": "#4a7ebb", "model-detected": "#e08a2c"}
    tp = {pop: ris_tab[ris_tab.population == pop].set_index("land_cover") for pop, _ in POPS}
    x = np.arange(len(show))
    wbar = 0.38
    fig, axes = plt.subplots(3, 1, figsize=(max(7.5, 1.15 * len(show) + 2), 10), sharex=True)
    for ax, p in zip(axes, PROFILES):
        for j, (pop, _) in enumerate(POPS):
            sub = tp[pop].reindex(show)
            xpos = x + (j - 0.5) * wbar
            ax.bar(xpos, sub["RIS_%s_mean" % p], width=wbar, color=POP_COLOR[pop], label=pop + " - mean")
            ax.scatter(xpos, sub["RIS_%s_median" % p], marker="D", s=14, color="k", zorder=3,
                       label="median" if j == 1 else None)
            for xi, sm in zip(xpos, sub.small_n):
                if sm is True:
                    ax.text(xi, 0.02, "*", ha="center", fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_ylabel("RIS")
        ax.set_title(PROFILE_LABEL[p] + " profile", fontsize=10, loc="left")
        ax.grid(axis="y", alpha=0.25)
    axes[0].legend(fontsize=8, ncol=3, loc="upper right")
    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(["{}\nref {:,} / det {:,}".format(c, int(tp["documented-burned"].n_px.get(c, 0)),
                                                              int(tp["model-detected"].n_px.get(c, 0))) for c in show],
                             fontsize=7.5)
    fig.suptitle("RIS by 2018 land cover, 30 m pixels: documented-burned vs model-detected "
                 "(label = pixel counts; * n < {} px)\n{}".format(MIN_PX_CLASS, FRAMING), fontsize=9.5)
    fig.tight_layout()
    save_figure(fig, "f01_ris_by_landcover_grouped_bars",
                "Mean (bars) and median (diamonds) RIS per 2018 land-cover class for the three weight profiles, "
                "over documented-burned and model-detected pixels. " + FRAMING, section=4, sources=SRC_1)
    plt.show()

    for pop in ("documented-burned", "model-detected"):
        s = tp[pop][~tp[pop].small_n & ~tp[pop].index.isin(["all classes", OTHER])].RIS_balanced_mean
        output_note("t01_ris_by_landcover" if pop == "documented-burned" else "f01_ris_by_landcover_grouped_bars",
                    "Balanced RIS mean over {} pixels: {} {:.3f} (lowest) to {} {:.3f} (highest), all classes {:.3f}; "
                    "{} classes with n >= {} px.".format(pop, s.idxmin(), s.min(), s.idxmax(), s.max(),
                                                        tp[pop].loc["all classes", "RIS_balanced_mean"], len(s), MIN_PX_CLASS))

## 5. Deliverable 2 — regional hierarchy: average per burned zone, then generalise per class

**Question.** Does averaging first within each burned area and then across areas (the two-level average) give a different land-cover picture from pooling all pixels?

**Why.** A pixel-pooled class mean is dominated by the few largest burned areas; the two-level average weights every burned area equally, which is the "average per area, then generalise" summary requested for the Results chapter.

**Data.** Burned zones from §3 (8-connected components of the documented-burned mask); the indicator and RIS layers; `landcover_2018_30m.tif`. Flat per-pixel means are taken directly from deliverable 1 (table t01, documented-burned), and the cell asserts that they match the pixels used here.

**Metric.** Level 1: the mean of each indicator and RIS profile over the pixels of class *c* inside zone *z* (a zone × class unit), keeping units of at least 4 pixels (0.36 ha). Level 2: the mean (and median) of those unit means per class, with `n_zones` = the number of zones contributing to that class. Because most zones contain several classes, one zone can count towards several classes. As a second view, each zone is assigned to its dominant class and its whole-zone mean averaged per class (`dominant_zone_mean`, `n_zones_dominant`). `zone_minus_pixel` is the area-level minus the pixel-level mean.

**Reading the output.** A large `zone_minus_pixel` means the class's pixel mean is driven by a few big zones. Either way, cross-class differences remain subject to the Notebook 13 normalisation caveat.

In [ ]:
if not HAVE_LC:
    print("SKIPPED — deliverable 2 needs data/raw/landcover_2018_30m.tif aligned to the inference grid (see §2).")
else:
    pix = pd.DataFrame({"zone": ZID, "cls": CLS[REF]})
    for q in QUANT:
        pix[q] = Q[q][REF]
    unit = pix.groupby(["zone", "cls"]).agg(n_px=("zone", "size"), **{q: (q, "mean") for q in QUANT}).reset_index()
    unit_ok = unit[unit.n_px >= MIN_UNIT_PX]
    zn = pix.groupby("zone").size()
    zdom = (pix.groupby(["zone", "cls"]).size().rename("n").reset_index()
            .sort_values(["zone", "n", "cls"], ascending=[True, False, True])
            .drop_duplicates("zone").set_index("zone"))
    zlev = pix.groupby("zone")[QUANT].mean().assign(n_px=zn, cls=zdom["cls"], dominant_share=zdom["n"] / zn)
    zlev_ok = zlev[zlev.n_px >= MIN_UNIT_PX]

    flat = tp["documented-burned"]
    for ci, v in pix.groupby("cls")["RIS_balanced"].mean().items():     # flat means reused from t01, verified
        assert np.isclose(v, flat.loc[CLASS_NAMES[ci], "RIS_balanced_mean"]), CLASS_NAMES[ci]

    rows = []
    for ci, cname in list(enumerate(CLASS_NAMES)) + [(None, "all classes")]:
        if cname not in flat.index:
            continue
        u = zlev_ok if ci is None else unit_ok[unit_ok.cls == ci]
        ud = zlev_ok if ci is None else zlev_ok[zlev_ok.cls == ci]
        n_flat = int(flat.loc[cname, "n_px"])
        for q in QUANT:
            fm = float(flat.loc[cname, q + "_mean"])
            zl = float(u[q].mean()) if len(u) else np.nan
            rows.append(dict(land_cover=cname, quantity=q, small_n=bool(flat.loc[cname, "small_n"]),
                             n_px=n_flat, flat_pixel_mean=fm,
                             n_zones=int(len(u)), pct_px_in_counted_zones=100.0 * float(u.n_px.sum()) / n_flat,
                             zone_level_mean=zl, zone_level_median=float(u[q].median()) if len(u) else np.nan,
                             zone_minus_pixel=zl - fm,
                             n_zones_dominant=int(len(ud)),
                             dominant_zone_mean=float(ud[q].mean()) if len(ud) else np.nan))
    hier = pd.DataFrame(rows)
    hier["_o"] = hier.land_cover.map({c: i for i, c in enumerate(order)})
    hier["_q"] = hier.quantity.map({q: i for i, q in enumerate(QUANT)})
    hier = hier.sort_values(["_q", "_o"]).drop(columns=["_o", "_q"]).reset_index(drop=True)
    display(hier[hier.quantity == "RIS_balanced"])

    SRC_2 = {"zones": "8-connected components of the documented-burned mask (fig_gt_fires.shp 2017-2025 on the 30 m grid)",
             "land_cover": "data/raw/landcover_2018_30m.tif",
             "flat_pixel_mean / n_px": "nb14_t01_ris_by_landcover (documented-burned rows), from the indicator rasters",
             "zone_level_* / dominant_zone_mean": "zone x class and whole-zone means of <indicator>_30m.tif and RIS",
             "quantity RIS_*": "weighted sum of severity/persistence/recovery_gap/model_recurrence _30m.tif"}
    save_table(hier, "t04_regional_hierarchy_zone_vs_pixel",
               "Two-level (area) average versus flat pixel average per 2018 land-cover class, documented-burned "
               "pixels: zone x class means (units >= {} px) averaged per class, the dominant-class alternative, and "
               "the flat pixel mean from deliverable 1. ".format(MIN_UNIT_PX) + FRAMING, section=5, sources=SRC_2)

    show2 = [c for c in ref_order if not flat.loc[c, "small_n"]] + ["all classes"]
    yy = np.arange(len(show2))[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 0.5 * len(show2) + 2), sharey=True)
    for ax, q, ttl in zip(axes, ["RIS_balanced", "recovery_gap"], ["RIS, Balanced profile", "recovery gap (model)"]):
        sub = hier[hier.quantity == q].set_index("land_cover").reindex(show2)
        ax.scatter(sub.flat_pixel_mean, yy, marker="o", color="k", s=30, zorder=3, label="pixel level (flat mean)")
        ax.scatter(sub.zone_level_mean, yy, marker="s", color="tab:red", s=30, zorder=3, label="area level: zone × class means")
        ax.scatter(sub.dominant_zone_mean, yy, marker="^", color="tab:blue", s=30, zorder=3, label="area level: zones by dominant class")
        xmax = sub[["flat_pixel_mean", "zone_level_mean", "dominant_zone_mean"]].max(axis=1)
        for yi, nz, nzd, xm in zip(yy, sub.n_zones, sub.n_zones_dominant, xmax):
            ax.text(min(xm + 0.02, 0.8), yi, "{} / {} zones".format(int(nz), int(nzd)), va="center", fontsize=7)
        ax.set_xlim(0, 1)
        ax.set_title(ttl, fontsize=10)
        ax.set_xlabel("class mean")
        ax.grid(axis="x", alpha=0.25)
    axes[0].set_yticks(yy)
    axes[0].set_yticklabels(show2)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle("Pixel-level vs area-level (burned-zone) class means, documented-burned pixels "
                 "(label: zone × class units / dominant-class zones)\n" + FRAMING, fontsize=9.5)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    save_figure(fig, "f02_regional_hierarchy_zone_vs_pixel",
                "Flat pixel mean versus two-level zone means per 2018 land-cover class for Balanced RIS and the "
                "recovery gap. " + FRAMING, section=5, sources=SRC_2)
    plt.show()

    s = hier[(hier.quantity == "RIS_balanced") & ~hier.small_n & (hier.land_cover != "all classes")].set_index("land_cover")
    a_ = hier[(hier.quantity == "RIS_balanced") & (hier.land_cover == "all classes")].iloc[0]
    output_note("t04_regional_hierarchy_zone_vs_pixel",
                "Balanced RIS, area-level minus pixel-level: {:+.3f} ({}) to {:+.3f} ({}); zones per class {} to {}; "
                "all classes: pixel {:.3f} vs zone {:.3f} over {:,} zones."
                .format(s.zone_minus_pixel.min(), s.zone_minus_pixel.idxmin(), s.zone_minus_pixel.max(),
                        s.zone_minus_pixel.idxmax(), int(s.n_zones.min()), int(s.n_zones.max()),
                        a_.flat_pixel_mean, a_.zone_level_mean, int(a_.n_zones)))

## 6. Deliverable 3 — restoration priority over documented-burned vs model-detected areas

**Question.** Where does the model's restoration priority fall when displayed over the reference fire inventory, and where when displayed over the model's own detections?

**Why.** The RIS surface is the same in both maps; only the mask differs. With an identical colormap and scale, what differs between the maps is detection agreement and disagreement.

**Data.** Balanced RIS (§3); documented-burned and model-detected masks (§3); ROI outline `fig_roi_polygon.shp`.

**Metric.** Full region at 30 m; light gray = eligible area with no burned/detected pixel. Titles give pixel counts, area and RIS mean/median per mask; the note gives the overlap.

**Reading the output.** Colour in (b) but not (a) = detections outside the (incomplete) inventory; colour in (a) but not (b) = inventory fires the model did not flag.

In [ ]:
roi = gpd.read_file(ROI_SHP).to_crs(CRS)
extent = [GRID_T.c, GRID_T.c + W * GRID_T.a, GRID_T.f + H * GRID_T.e, GRID_T.f]
bg = np.where(ELIG, 1.0, np.nan)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 9.4), sharex=True, sharey=True, layout="constrained")
for ax, (lab, M) in zip(axes, [("(a) documented-burned areas (fire-reference polygons 2017-2025)", REF),
                               ("(b) model-detected areas (burned probability > 0.502)", DET)]):
    ax.imshow(bg, extent=extent, cmap=ListedColormap(["#e9e9e9"]), interpolation="nearest")
    im = ax.imshow(np.where(M, RIS["balanced"], np.nan), extent=extent, cmap=RESTORATION_CMAP,
                   vmin=0, vmax=1, interpolation="nearest")
    roi.boundary.plot(ax=ax, color="k", linewidth=0.6)
    mu, md = _ris_stats(M)
    ax.set_title("{}\n{:,} px ({:,.0f} ha); RIS mean {:.2f}, median {:.2f}".format(lab, int(M.sum()), M.sum() * PX_HA, mu, md),
                 fontsize=9.5)
    km_axes(ax)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
axes[1].set_ylabel("")
cb = fig.colorbar(im, ax=list(axes), orientation="horizontal", fraction=0.03, pad=0.02, shrink=0.6)
cb.set_label("Restoration priority - RIS, Balanced profile (identical colormap and 0-1 scale in both maps)")
fig.suptitle("Restoration priority: documented-burned vs model-detected areas - overlap {:,} px, Jaccard {:.2f}"
             .format(int(BOTH.sum()), jac), fontsize=11)
save_figure(fig, "f03_ris_balanced_reference_vs_detected_maps",
            "Balanced RIS at 30 m over (a) pixels inside the 2017-2025 fire-reference polygons and (b) pixels the "
            "model detects as burned (burned_prob > 0.502). Same RIS surface, colormap and 0-1 scale; gray = eligible "
            "area. The reference inventory is incomplete.", section=6,
            sources={"RIS_balanced": "0.25 x (severity + persistence + recovery_gap + model_recurrence) _30m.tif",
                     "mask (a)": "data/thesis_maps/fig_gt_fires.shp, fire_year 2017-2025, on the 30 m grid",
                     "mask (b)": "outputs/inference_full_roi/burned_prob_30m.tif > 0.502",
                     "outline": "data/thesis_maps/fig_roi_polygon.shp"})
plt.show()
m_ro, _ = _ris_stats(REF & ~DET)
m_do, _ = _ris_stats(DET & ~REF)
m_b, _ = _ris_stats(BOTH)
output_note("f03_ris_balanced_reference_vs_detected_maps",
            "(a) {:,} px, RIS mean {:.3f}; (b) {:,} px, RIS mean {:.3f}; shared {:,} px (Jaccard {:.3f}, RIS {:.3f}); "
            "reference-only {:,} px (RIS {:.3f}); detected-only {:,} px (RIS {:.3f})."
            .format(int(REF.sum()), _ris_stats(REF)[0], int(DET.sum()), _ris_stats(DET)[0], int(BOTH.sum()), jac, m_b,
                    int((REF & ~DET).sum()), m_ro, int((DET & ~REF).sum()), m_do))

## 7. Deliverable 4 — per-land-cover-class maps (small multiples)

**Question.** What do the RIS and the indicators look like inside each major land-cover class alone?

**Why.** Masking the maps to one class at a time, on a shared colour scale, makes the per-class level shifts visible — which Notebook 13 traced to the NBR baseline normalisation rather than to recovery.

**Data.** Balanced RIS, recovery gap, severity and persistence (§3); `landcover_2018_30m.tif`; both masks. Model recurrence is not mapped: it takes only the values 0.2-1.0 in steps of 0.2 and is 0.2 for most detected pixels. Panel annotations (n and mean) are read from table t01, not recomputed.

**Metric.** One figure per quantity; columns = land-cover classes with at least 100 documented-burned pixels, ordered by Balanced-RIS mean; rows = documented-burned and model-detected. Gray = that class's eligible footprint. All panels are cropped to the bounding box of all burned or detected pixels and share the 0-1 scale. **Display only:** each panel is drawn at 90 m, every cell being the mean of that class's selected 30 m pixels in its 3 × 3 block — at small-multiple size a 30 m map is downsampled about three-fold and isolated burned pixels would otherwise vanish. All numbers (panel n and mean, and every table) are 30 m pixel statistics.

**Reading the output.** A uniformly warmer or cooler panel is a class-level shift of the index, not a map of recovery.

In [ ]:
if not HAVE_LC:
    print("SKIPPED — deliverable 4 needs data/raw/landcover_2018_30m.tif aligned to the inference grid (see §2).")
else:
    major = [c for c in ref_order if not tp["documented-burned"].loc[c, "small_n"]]
    DISP = 3                                   # display-only 3 x 3 blocks = 90 m cells

    def block_mean(a):
        hh, ww = a.shape
        return np.nanmean(a.reshape(hh // DISP, DISP, ww // DISP, DISP), axis=(1, 3))

    Rr, Cc = np.nonzero(REF | DET)
    r0, c0 = max(int(Rr.min()) - 20, 0), max(int(Cc.min()) - 20, 0)
    r1 = r0 + ((min(int(Rr.max()) + 21, H) - r0) // DISP) * DISP
    c1 = c0 + ((min(int(Cc.max()) + 21, W) - c0) // DISP) * DISP
    crop_ext = [GRID_T.c + c0 * GRID_T.a, GRID_T.c + c1 * GRID_T.a, GRID_T.f + r1 * GRID_T.e, GRID_T.f + r0 * GRID_T.e]
    for q, qlabel in [("RIS_balanced", "RIS, Balanced profile"), ("recovery_gap", "Recovery gap (model)"),
                      ("severity", "Severity (model)"), ("persistence", "Persistence (model)")]:
        k = len(major)
        fig, axes = plt.subplots(2, k, figsize=(2.3 * k + 0.6, 8.4), sharex=True, sharey=True, squeeze=False,
                                 layout="constrained")
        for j, cname in enumerate(major):
            cm = CLS == CLASS_NAMES.index(cname)
            foot = block_mean(np.where(ELIG & cm, 1.0, np.nan)[r0:r1, c0:c1])
            for i, (pop, M) in enumerate(POPS):
                ax = axes[i, j]
                ax.imshow(foot, extent=crop_ext, cmap=ListedColormap(["#dcdcdc"]), interpolation="nearest")
                im = ax.imshow(block_mean(np.where(M & cm, Q[q], np.nan)[r0:r1, c0:c1]), extent=crop_ext,
                               cmap=RESTORATION_CMAP, vmin=0, vmax=1, interpolation="nearest")
                t = tp[pop]
                if cname in t.index:
                    txt = "n = {:,}\nmean = {:.2f}".format(int(t.loc[cname, "n_px"]), t.loc[cname, q + "_mean"])
                else:
                    txt = "n = 0"
                ax.text(0.03, 0.03, txt, transform=ax.transAxes, fontsize=6.5, va="bottom",
                        bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_frame_on(False)
                if i == 0:
                    ax.set_title(cname, fontsize=9, color=CLASS_COLOR[cname])
        axes[0, 0].set_ylabel("documented-burned", fontsize=9)
        axes[1, 0].set_ylabel("model-detected", fontsize=9)
        cb = fig.colorbar(im, ax=axes.ravel().tolist(), orientation="horizontal", fraction=0.04, pad=0.02, shrink=0.5)
        cb.set_label(qlabel + " - one shared 0-1 scale for every panel")
        fig.suptitle("{} within each 2018 land-cover class (gray = class footprint; drawn as 90 m block means, "
                     "statistics at 30 m)\n{}".format(qlabel, FRAMING), fontsize=9.5)
        stem = "f04_{}_by_class_small_multiples".format(q)
        save_figure(fig, stem,
                    "{} shown only inside each major 2018 land-cover class, over documented-burned (top) and model-detected "
                    "(bottom) pixels, shared 0-1 scale, drawn as 90 m block means of the 30 m class pixels (display only). "
                    "Panel n and mean are 30 m statistics from nb14_t01_ris_by_landcover. {}".format(qlabel, FRAMING),
                    section=7, sources={q: ("RIS from the four indicator rasters" if q.startswith("RIS")
                                            else "outputs/inference_full_roi/{}_30m.tif".format(q)),
                                        "class mask": "data/raw/landcover_2018_30m.tif",
                                        "n / mean annotations": "nb14_t01_ris_by_landcover",
                                        "masks": "fig_gt_fires.shp (2017-2025) on the 30 m grid; burned_prob_30m.tif > 0.502"})
        plt.show()
        sr = tp["documented-burned"].loc[major, q + "_mean"]
        output_note(stem, "{} class means over documented-burned pixels: {} {:.3f} to {} {:.3f} across {} classes (shared 0-1 scale)."
                    .format(qlabel, sr.idxmin(), sr.min(), sr.idxmax(), sr.max(), len(major)))

## 8. Result registry and output notes

In [ ]:
REGISTRY.append(dict(section="8", kind="table", stem=PREFIX + "t99_result_registry",
                     files=_rel(os.path.join(TAB, PREFIX + "t99_result_registry")) + ".csv, .md",
                     sources="REGISTRY", caption="This registry."))
registry = pd.DataFrame(REGISTRY)[["section", "kind", "stem", "files", "sources", "caption"]]
registry.to_csv(os.path.join(TAB, PREFIX + "t99_result_registry.csv"), index=False)
with open(os.path.join(TAB, PREFIX + "t99_result_registry.md"), "w") as fh:
    fh.write("# Result registry — notebooks/14_landuse_presentation.ipynb\n\nGenerated {}{}.\n\n"
             .format(RUN_STAMP, " (land-cover raster missing: §4, §5, §7 skipped)" if not HAVE_LC else ""))
    fh.write(_markdown_table(registry.drop(columns=["caption"])))
    fh.write("\n## Output notes\n\n" + "\n".join("- " + n for n in NOTES) + "\n")
display(registry.drop(columns=["caption"]))
print("\n".join(NOTES))